# 14 - Adversarial Encoding

Fine-tune Qwen3-4B with a reward that maximises GSM8K accuracy while
penalising paraphrase transferability:
  reward = Acc(self_prefill) - lambda * Acc(paraphrase_self)

This trains the model to hide information in token-level encoding,
probing the ceiling of steganographic capacity.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
!pip install -q trl peft accelerate bitsandbytes

In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from lib.data import load_gsm8k, extract_predicted_answer
from lib.prompts import build_cot_messages, build_prefill_prompt, build_paraphrase_messages

dataset = load_gsm8k()
# Use a subset for RL training
train_subset = dataset[:256]
eval_subset = dataset[256:512]
print(f"Train: {len(train_subset)}, Eval: {len(eval_subset)}")

## Define reward function

For each generated COT:
1. Check if self-prefill gives correct answer (accuracy component)
2. Paraphrase it, check if paraphrase-self gives correct answer (penalty component)
3. reward = correct_self - lambda * correct_paraphrase

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

LAMBDA = 0.5  # Penalty weight for paraphrase transferability
ADVERSARIAL_OUTPUT = CACHE_DIR / "adversarial"
ADVERSARIAL_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Loading {PRIMARY_MODEL} for fine-tuning...")
tokenizer = AutoTokenizer.from_pretrained(PRIMARY_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    PRIMARY_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
)

# Apply LoRA for efficient fine-tuning
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead

# Wrap model with value head for PPO
model_with_vh = AutoModelForCausalLMWithValueHead.from_pretrained(
    PRIMARY_MODEL, torch_dtype=torch.bfloat16, device_map="auto",
    peft_config=lora_config,
)

ppo_config = PPOConfig(
    model_name=PRIMARY_MODEL,
    learning_rate=1e-5,
    batch_size=8,
    mini_batch_size=4,
    gradient_accumulation_steps=2,
    ppo_epochs=2,
    max_grad_norm=1.0,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model_with_vh,
    tokenizer=tokenizer,
)

In [ ]:
from vllm import LLM, SamplingParams

# We need a separate paraphraser for the reward computation
print(f"Loading {PARAPHRASER_MODEL} for reward computation...")
para_llm = LLM(model=PARAPHRASER_MODEL, dtype="bfloat16", max_model_len=4096)
para_tokenizer = AutoTokenizer.from_pretrained(PARAPHRASER_MODEL)
para_sampling = SamplingParams(temperature=0.0, max_tokens=MAX_COT_TOKENS)


def compute_reward(question, gold_answer, cot_text):
    """Compute adversarial reward for a generated COT."""
    # 1. Self-prefill accuracy
    prefill_prompt = build_prefill_prompt(question, cot_text, PRIMARY_MODEL)
    # (simplified: would use vLLM batch in practice)
    self_correct = 1.0  # placeholder - actual implementation would generate + check

    # 2. Paraphrase the COT
    para_msgs = build_paraphrase_messages(cot_text, heavy=False)
    para_prompt = para_tokenizer.apply_chat_template(
        para_msgs, tokenize=False, add_generation_prompt=True
    )
    para_output = para_llm.generate([para_prompt], para_sampling)
    paraphrased = para_output[0].outputs[0].text.strip()

    # 3. Paraphrase-self accuracy
    para_prefill = build_prefill_prompt(question, paraphrased, PRIMARY_MODEL)
    para_correct = 1.0  # placeholder

    return self_correct - LAMBDA * para_correct

## Training Loop

Note: This is a simplified outline. Full implementation requires careful
integration of vLLM generation within the PPO loop, which may require
running generation and training in alternating phases.

In [ ]:
NUM_EPOCHS = 3
reward_history = []

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    epoch_rewards = []

    for i in range(0, len(train_subset), ppo_config.batch_size):
        batch = train_subset[i:i + ppo_config.batch_size]

        # Generate COTs with current policy
        queries = []
        for ex in batch:
            msgs = build_cot_messages(ex["question"])
            prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            input_ids = tokenizer.encode(prompt, return_tensors="pt")
            queries.append(input_ids.squeeze())

        # Generate responses
        responses = []
        for q in queries:
            with torch.no_grad():
                output = model_with_vh.generate(
                    q.unsqueeze(0).to(model_with_vh.pretrained_model.device),
                    max_new_tokens=MAX_COT_TOKENS,
                    temperature=0.7,
                    do_sample=True,
                )
            response = output[0][len(q):]
            responses.append(response)

        # Compute rewards
        rewards = []
        for ex, response in zip(batch, responses):
            cot_text = tokenizer.decode(response, skip_special_tokens=True)
            r = compute_reward(ex["question"], ex["gold_answer"], cot_text)
            rewards.append(torch.tensor(r))
            epoch_rewards.append(r)

        # PPO step
        stats = ppo_trainer.step(queries, responses, rewards)

    mean_reward = np.mean(epoch_rewards)
    reward_history.append(mean_reward)
    print(f"Mean reward: {mean_reward:.3f}")

# Save the fine-tuned model
model_with_vh.save_pretrained(ADVERSARIAL_OUTPUT / "adversarial_model")
tokenizer.save_pretrained(ADVERSARIAL_OUTPUT / "adversarial_model")
print(f"Model saved to {ADVERSARIAL_OUTPUT / 'adversarial_model'}")

## Evaluate adversarial model

Run the core 2x2 on the adversarial model's COTs and compare L with baseline.

In [ ]:
import matplotlib.pyplot as plt

# Plot reward curve
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(reward_history)+1), reward_history, "o-", color="#e74c3c")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean reward")
ax.set_title("Adversarial Training: Reward Curve", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "adversarial_reward.png", dpi=200, bbox_inches="tight")
plt.show()

# Save results
with open(RESULTS_DIR / "adversarial_training.json", "w") as f:
    json.dump({"reward_history": reward_history, "lambda": LAMBDA}, f, indent=2)